# 사용할 코드만 정리

In [1]:
# 도구 불러오기
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from glob import glob
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.font_manager as fm

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

# 1. 한글 폰트 설정 (Windows의 경우 맑은 고딕 사용)
plt.rcParams['font.family'] = 'Malgun Gothic'
# 2. 마이너스 기호(-)가 깨지는 현상 방지
plt.rcParams['axes.unicode_minus'] = False
# (옵션) 그래프의 글자를 좀 더 선명하게 보고 싶다면
# %config InlineBackend.figure_format = 'retina'

In [2]:
# 원본 메타데이터 로드
df_meta = pd.read_csv("../../dataset/metadata.csv")

In [3]:
# 전처리 (시간/수치형(Capacity, Re, Rct)파생)

def clean_parse_time(x):
    if pd.isna(x):
        return pd.NaT
    
    try:
        s = str(x)
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if len(parts) < 3:
            return pd.NaT
        
        nums = [float(p) for p in parts]
        
        year   = int(nums[0])
        month  = int(nums[1])
        day    = int(nums[2])
        hour   = int(nums[3]) if len(nums) > 3 else 0
        minute = int(nums[4]) if len(nums) > 4 else 0
        
        # 핵심: round 제거
        second = int(nums[5]) if len(nums) > 5 else 0
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hour, minute=minute, second=second)
    
    except Exception as e:
        print(f"[파싱 실패] {x} | {e}")
        return pd.NaT

def clean_numeric(x):
    """수치형 데이터의 대괄호 제거 및 float 변환"""
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [전처리 실행] ---
df = df_meta.copy()

# 1. 시간 데이터 정제 (소수점 초까지 완벽 대응)
df['start_time'] = df['start_time'].apply(clean_parse_time)

# 2. 수치형 데이터 정제 (Capacity, Re, Rct)
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 물리적 순서 정렬
df = df.sort_values(['battery_id', 'uid']).reset_index(drop=True)

# 최종 결과 검증
print(f"전체 데이터 개수: {len(df):,}개")
print(f"시간 결측치(NaT): {df['start_time'].isna().sum()}개")
print(f"수치형 결측치(Cap): {df['Capacity'].isna().sum()}개")


전체 데이터 개수: 7,565개
시간 결측치(NaT): 0개
수치형 결측치(Cap): 4796개


# metadata df에서 B0005만 필터링

In [4]:
# metadata df에서 B0005만 필터링하여 시간순 정렬

# 판다스 출력 옵션 설정 (최대 행/열 제한 해제)
pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.width', None)  # 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None) # 컬럼 내용 생략 없이 출력

# 생성
B0005_timeline = df[df['battery_id'] == 'B0005'].copy()

# 시간순 정렬 후 주요 컬럼만 출력
# 'type', 'start_time', 'cycle'을 중심으로 나열
check_cols = ['type', 'start_time', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity','Re','Rct', 'ambient_temperature']
display(B0005_timeline[check_cols].sort_values('start_time').head(50))

# 출력이 끝난 후 옵션을 다시 기본값으로 되돌리고 싶다면 (선택사항)
# pd.reset_option('all')

,type,start_time,battery_id,test_id,uid,filename,Capacity,Re,Rct,ambient_temperature
0,charge,2008-04-02 13:08:17,B0005,0,5121,05121.csv,NaN,NaN,NaN,24
1,discharge,2008-04-02 15:25:41,B0005,1,5122,05122.csv,1.856487,NaN,NaN,24
2,charge,2008-04-02 16:37:51,B0005,2,5123,05123.csv,NaN,NaN,NaN,24
3,discharge,2008-04-02 19:43:48,B0005,3,5124,05124.csv,1.846327,NaN,NaN,24
4,charge,2008-04-02 20:55:40,B0005,4,5125,05125.csv,NaN,NaN,NaN,24
5,discharge,2008-04-03 00:01:06,B0005,5,5126,05126.csv,1.835349,NaN,NaN,24
6,charge,2008-04-03 01:12:38,B0005,6,5127,05127.csv,NaN,NaN,NaN,24
7,discharge,2008-04-03 04:16:37,B0005,7,5128,05128.csv,1.835263,NaN,NaN,24
8,charge,2008-04-03 05:27:49,B0005,8,5129,05129.csv,NaN,NaN,NaN,24
9,discharge,2008-04-03 08:33:25,B0005,9,5130,05130.csv,1.834646,NaN,NaN,24


# B0005_timeline의 메타정보를 data들을 가져와서 병합

In [5]:
# B0005_timeline의 메타정보를 data들을 가져와서 병합

# 1. 설정 및 타임라인 준비
target_id = 'B0005'
data_folder = "../../dataset/data" 

# 원본 타임라인 데이터프레임을 B0005_timeline으로 정의하고 test_id
B0005_timeline.rename(columns={'test_id': 'test_id'}, inplace=True)

# 2. EOL 계산 (RUL 산출용)
dis_only = B0005_timeline[B0005_timeline['type'] == 'discharge']
first_cap = dis_only['Capacity'].iloc[0] if not dis_only.empty else 0
eol_idx = np.where((dis_only['Capacity'] / first_cap) * 100 <= 80)[0]
eol_cycle_num = eol_idx[0] + 1 if len(eol_idx) > 0 else np.nan

# 3. 통합 저장소 및 카운터
all_data_list = []
counters = {'charge': 1, 'discharge': 1, 'impedance': 1}
discharge_cycle_counter = 0

# 4. 루프 및 조인 (B0005_timeline 기준)
for _, row in B0005_timeline.iterrows():
    file_path = os.path.join(data_folder, row['filename'])
    
    if os.path.exists(file_path):
        # CSV 로드
        temp_df = pd.read_csv(file_path)
        d_type = row['type']
        
        # (A) 타임라인의 모든 메타데이터 주입 (test_id 포함)
        for col in B0005_timeline.columns:
            temp_df[col] = row[col]
        
        # (B) 타입별 내부 회차 (cycle_in_type)
        temp_df['cycle_in_type'] = counters[d_type]
        
        # (C) 방전 기준 사이클 (discharge_cycle)
        if d_type == 'discharge':
            discharge_cycle_counter += 1
        temp_df['discharge_cycle'] = discharge_cycle_counter
        temp_df['global_cycle'] = discharge_cycle_counter
        
        # (D) SOH & (E) RUL
        temp_df['SOH_target'] = (row['Capacity'] / first_cap * 100) if d_type == 'discharge' else np.nan
        if pd.notnull(eol_cycle_num):
            temp_df['RUL'] = max(0, eol_cycle_num - discharge_cycle_counter)
        else:
            temp_df['RUL'] = np.nan
        
        all_data_list.append(temp_df)
        
        # 카운터 업데이트
        counters[d_type] += 1
    else:
        print(f"[누락 파일] {file_path}")

# 5. 최종 합치기
df_total_B0005 = pd.concat(all_data_list, ignore_index=True)

# 6. 시간순 정렬 및 결측치 처리
df_total_B0005['start_time'] = pd.to_datetime(df_total_B0005['start_time'])
df_total_B0005 = df_total_B0005.sort_values(['start_time', 'test_id', 'Time'])

# SOH_target ffill/bfill
df_total_B0005['SOH_target'] = df_total_B0005['SOH_target'].ffill()

print(f"✅ 통합 완료! B0005_timeline의 인덱스가 'test_id'로 반영되었습니다.")
print(f"포함된 컬럼: {df_total_B0005.columns.tolist()}")

✅ 통합 완료! B0005_timeline의 인덱스가 'test_id'로 반영되었습니다.
포함된 컬럼: ['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time', 'type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct', 'cycle_in_type', 'discharge_cycle', 'global_cycle', 'SOH_target', 'RUL', 'Current_load', 'Voltage_load', 'Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance']


# 방전시간과 방전시작시간~임피던스 시간이 일치하는지 확인하기

In [6]:
# 데이터프레임 구성 1번째- B0005_timeline으로 시계열 구성하기

# 1. 데이터 정렬
df = B0005_timeline.copy()
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.sort_values('start_time').reset_index(drop=True)

# 타입별 인덱스 미리 추출
chg_idx = df[df['type'] == 'charge'].index
dis_idx = df[df['type'] == 'discharge'].index
imp_idx = df[df['type'] == 'impedance'].index

extracted_rows = []

# 2. 모든 '충전(charge)'을 시작점으로 루프
for i in range(len(chg_idx)):
    curr_chg_idx = chg_idx[i]
    next_chg_idx = chg_idx[i+1] if i+1 < len(chg_idx) else None
    
    # 현재 충전 정보
    row = {
        'chg_t_cycle': df.loc[curr_chg_idx, 'test_id'],
        'chg_start': df.loc[curr_chg_idx, 'start_time']
    }
    
    # [dis] 현재 충전과 다음 충전 사이에 방전이 있는가?
    upper_bound = next_chg_idx if next_chg_idx is not None else len(df)
    dis_in_between = dis_idx[(dis_idx > curr_chg_idx) & (dis_idx < upper_bound)]
    
    if len(dis_in_between) > 0:
        d_idx = dis_in_between[0]
        row['dis_t_cycle'] = df.loc[d_idx, 'test_id']
        row['dis_start'] = df.loc[d_idx, 'start_time']
        
        # [imp] chg와 dis 사이에 impedance가 있는가?
        imp_mid = imp_idx[(imp_idx > curr_chg_idx) & (imp_idx < d_idx)]
        if len(imp_mid) > 0:
            row['imp_t_cycle'] = df.loc[imp_mid[0], 'test_id']
            row['imp_start'] = df.loc[imp_mid[0], 'start_time']
        else:
            row['imp_t_cycle'], row['imp_start'] = np.nan, np.nan
            
        # [imp2] dis와 다음 chg 사이에 impedance가 있는가?
        imp_post = imp_idx[(imp_idx > d_idx) & (imp_idx < upper_bound)]
        if len(imp_post) > 0:
            row['imp2_t_cycle'] = df.loc[imp_post[0], 'test_id']
            row['imp2_start'] = df.loc[imp_post[0], 'start_time']
        else:
            row['imp2_t_cycle'], row['imp2_start'] = np.nan, np.nan
            
        # [imp2_chg] 다음 충전 시간과의 차이 계산
        if next_chg_idx is not None and pd.notnull(row['imp2_start']):
            row['imp2_chg'] = (df.loc[next_chg_idx, 'start_time'] - row['imp2_start']).total_seconds()
        else:
            row['imp2_chg'] = np.nan
            
        extracted_rows.append(row)

# 3. 결과 생성
final_result = pd.DataFrame(extracted_rows)

# int64로 변환
cycle_cols = ['chg_t_cycle', 'imp_t_cycle', 'dis_t_cycle', 'imp2_t_cycle']
for col in cycle_cols:
    if col in final_result.columns:
        final_result[col] = final_result[col].astype('Int64')

# 컬럼 순서 정리
cols = ['chg_t_cycle', 'chg_start', 'imp_t_cycle', 'imp_start', 
        'dis_t_cycle', 'dis_start', 'imp2_t_cycle', 'imp2_start', 'imp2_chg']
final_result = final_result[cols]

display(final_result)

,chg_t_cycle,chg_start,imp_t_cycle,imp_start,dis_t_cycle,dis_start,imp2_t_cycle,imp2_start,imp2_chg
0,0,2008-04-02 13:08:17,<NA>,NaT,1,2008-04-02 15:25:41,<NA>,NaT,NaN
1,2,2008-04-02 16:37:51,<NA>,NaT,3,2008-04-02 19:43:48,<NA>,NaT,NaN
2,4,2008-04-02 20:55:40,<NA>,NaT,5,2008-04-03 00:01:06,<NA>,NaT,NaN
3,6,2008-04-03 01:12:38,<NA>,NaT,7,2008-04-03 04:16:37,<NA>,NaT,NaN
4,8,2008-04-03 05:27:49,<NA>,NaT,9,2008-04-03 08:33:25,<NA>,NaT,NaN
5,10,2008-04-03 09:44:35,<NA>,NaT,11,2008-04-03 12:55:10,<NA>,NaT,NaN
6,12,2008-04-03 14:06:43,<NA>,NaT,13,2008-04-03 17:17:16,<NA>,NaT,NaN
7,14,2008-04-03 18:28:47,<NA>,NaT,15,2008-04-03 21:28:14,<NA>,NaT,NaN
8,16,2008-04-03 22:38:27,<NA>,NaT,17,2008-04-04 01:38:15,<NA>,NaT,NaN
9,18,2008-04-04 02:48:06,<NA>,NaT,19,2008-04-04 05:48:08,<NA>,NaT,NaN


In [7]:
final_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   chg_t_cycle   167 non-null    Int64         
 1   chg_start     167 non-null    datetime64[ns]
 2   imp_t_cycle   136 non-null    Int64         
 3   imp_start     136 non-null    datetime64[ns]
 4   dis_t_cycle   167 non-null    Int64         
 5   dis_start     167 non-null    datetime64[ns]
 6   imp2_t_cycle  140 non-null    Int64         
 7   imp2_start    140 non-null    datetime64[ns]
 8   imp2_chg      140 non-null    float64       
dtypes: Int64(4), datetime64[ns](4), float64(1)
memory usage: 12.5 KB


In [17]:
# 데이터프레임 구성 2번째- B0005_timeline으로 구성된 시트(앞코드)에서 df_total_B0005를 매핑하여 전개함

# --- [보조 함수 1: Impedance 복소수 처리] ---
def parse_complex_cols(df, prefix):
    complex_cols = ['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance']
    result = {}
    
    for col in complex_cols:
        if col in df.columns and not df[col].empty:
            try:
                vals = df[col].dropna().apply(lambda x: complex(x) if isinstance(x, str) else x)
                vals = vals[vals.apply(lambda x: isinstance(x, complex))]
                
                if len(vals) > 0:
                    val = vals.mean()
                else:
                    val = np.nan
                    
            except:
                val = np.nan
        else:
            val = np.nan
        
        result[f"{prefix}_{col}_real"] = val.real if pd.notnull(val) else np.nan
        result[f"{prefix}_{col}_imag"] = val.imag if pd.notnull(val) else np.nan
    
    return result

# --- [보조 함수 2: 통계량 추출] ---
def get_stats(df, col, mode='basic'):
    if df is None or df.empty or col not in df.columns:
        if mode == 'temp': return [np.nan]*5
        if mode == 'full': return [np.nan]*3
        if mode == 'last': return np.nan
        return np.nan
    
    first = df[col].iloc[0]
    last = df[col].iloc[-1]
    diff = last - first
    avg = df[col].mean()
    mx = df[col].max()
    
    if mode == 'temp': return [first, avg, last, mx, diff]
    if mode == 'full': return [first, last, diff]
    if mode == 'last': return last
    return avg

# --- [메인 전개 프로세스] ---
final_rows = []

for i, row in final_result.iterrows():
    combined = {}
    
    # 1. Charge (cha_) 매핑
    c_data = df_total_B0005[df_total_B0005['test_id'] == row['chg_t_cycle']]
    combined['cha_t_cycle'] = row['chg_t_cycle']
    combined['cha_start'] = row['chg_start']
    combined['cha_time'] = get_stats(c_data, 'Time', 'last') # Time 끝값
    
    # Charge 파생 변수
    v_f, v_l, v_d = get_stats(c_data, 'Voltage_measured', 'full')
    combined.update({'cha_V_first': v_f, 'cha_V_last': v_l, 'cha_V_diff': v_d})
    i_f, i_l, i_d = get_stats(c_data, 'Current_measured', 'full')
    combined.update({'cha_I_first': i_f, 'cha_I_last': i_l, 'cha_I_diff': i_d})
    t_f, t_a, t_l, t_m, t_d = get_stats(c_data, 'Temperature_measured', 'temp')
    combined.update({'cha_T_first': t_f, 'cha_T_mean': t_a, 'cha_T_last': t_l, 'cha_T_max': t_m, 'cha_T_diff': t_d})
    vc_f, vc_l, vc_d = get_stats(c_data, 'Voltage_charge', 'full')
    combined.update({'cha_V_charge_first': vc_f, 'cha_V_charge_last': vc_l, 'cha_V_charge_diff': vc_d})
    ic_f, ic_l, ic_d = get_stats(c_data, 'Current_charge', 'full')
    combined.update({'cha_I_charge_first': ic_f, 'cha_I_charge_last': ic_l, 'cha_I_charge_diff': ic_d})

    # 2. Impedance 1 (imp_) 매핑
    i1_data = df_total_B0005[df_total_B0005['test_id'] == row['imp_t_cycle']]
    combined['imp_t_cycle'] = row['imp_t_cycle']
    combined['imp_start'] = row['imp_start']
    combined['imp1_Re'] = i1_data['Re'].iloc[0] if not i1_data.empty else np.nan
    combined['imp1_Rct'] = i1_data['Rct'].iloc[0] if not i1_data.empty else np.nan
    combined.update(parse_complex_cols(i1_data, 'imp'))

    # 3. Discharge (dis_) 매핑
    d_data = df_total_B0005[df_total_B0005['test_id'] == row['dis_t_cycle']]
    combined['dis_t_cycle'] = row['dis_t_cycle']
    combined['dis_start'] = row['dis_start']
    combined['dis_time'] = d_data['Time'].iloc[-1] if not d_data.empty else np.nan
    combined['SOH_target'] = d_data['SOH_target'].iloc[0] if not d_data.empty else np.nan
    combined['RUL'] = d_data['RUL'].iloc[0] if not d_data.empty else np.nan
    
    # Discharge 파생 변수
    dv_f, dv_l, dv_d = get_stats(d_data, 'Voltage_measured', 'full')
    combined.update({'dis_V_first': dv_f, 'dis_V_last': dv_l, 'dis_V_diff': dv_d})
    di_f, di_l, di_d = get_stats(d_data, 'Current_measured', 'full')
    combined.update({'dis_I_first': di_f, 'dis_I_last': di_l, 'dis_I_diff': di_d})
    dt_f, dt_a, dt_l, dt_m, dt_d = get_stats(d_data, 'Temperature_measured', 'temp')
    combined.update({'dis_T_first': dt_f, 'dis_T_mean': dt_a, 'dis_T_last': dt_l, 'dis_T_max': dt_m, 'dis_T_diff': dt_d})
    combined['dis_I_load_mean'] = get_stats(d_data, 'Current_load', 'avg')
    combined['dis_V_load_mean'] = get_stats(d_data, 'Voltage_load', 'avg')
    combined['dis_Capacity'] = d_data['Capacity'].iloc[0] if not d_data.empty else np.nan

    # 4. Impedance 2 (imp2_) 매핑
    i2_data = df_total_B0005[df_total_B0005['test_id'] == row['imp2_t_cycle']]
    combined['imp2_t_cycle'] = row['imp2_t_cycle']
    combined['imp2_start'] = row['imp2_start']
    combined['imp2_Re'] = i2_data['Re'].iloc[0] if not i2_data.empty else np.nan
    combined['imp2_Rct'] = i2_data['Rct'].iloc[0] if not i2_data.empty else np.nan
    combined.update(parse_complex_cols(i2_data, 'imp2'))

    # 5. Gap 계산 (요청하신 공식 기반)
    # cha_imp : imp_start - (cha_start + cha_time)
    if pd.notnull(combined['imp_start']) and pd.notnull(combined['cha_start']):
        combined['cha_imp'] = (combined['imp_start'] - (combined['cha_start'] + pd.Timedelta(seconds=combined['cha_time']))).total_seconds()
    else: combined['cha_imp'] = np.nan

    # imp_dis : dis_start - imp_start
    if pd.notnull(combined['dis_start']) and pd.notnull(combined['imp_start']):
        combined['imp_dis'] = (combined['dis_start'] - combined['imp_start']).total_seconds()
    else: combined['imp_dis'] = np.nan

    # dis_imp2 : imp2_start - (dis_start + dis_time)
    if pd.notnull(combined['imp2_start']) and pd.notnull(combined['dis_start']):
        combined['dis_imp2'] = (combined['imp2_start'] - (combined['dis_start'] + pd.Timedelta(seconds=combined['dis_time']))).total_seconds()
    else: combined['dis_imp2'] = np.nan

    # imp2_cha : '다음 행의 cha_start' - imp2_start (shift를 위해 나중에 처리하거나 여기서 미리 참조)
    # 루프 내에서는 계산이 어려우므로 아래에서 shift로 처리합니다.
    
    combined['global_cycle'] = i + 1 # 인덱스 기반 생성
    final_rows.append(combined)

# --- [최종 정리] ---
df_all = pd.DataFrame(final_rows)

# imp2_cha 계산 (다음 행의 cha_start 활용)
df_all['next_cha_start'] = df_all['cha_start'].shift(-1)
df_all['imp2_cha'] = (df_all['next_cha_start'] - df_all['imp2_start']).dt.total_seconds()

# 요청하신 최종 컬럼 순서 (순서에 맞춰 재정렬)
base_cols = ['cha_t_cycle', 'cha_start', 'cha_time']
charge_cols = [c for c in df_all.columns if c.startswith('cha_') and c not in base_cols and c != 'cha_imp']
imp1_cols = ['cha_imp', 'imp_t_cycle', 'imp_start', 'imp1_Re', 'imp1_Rct']
imp1_complex = [c for c in df_all.columns if c.startswith('imp_') and c not in imp1_cols and c != 'imp_dis']
dis_cols = ['imp_dis', 'dis_t_cycle', 'dis_start', 'dis_time']
discharge_cols = [c for c in df_all.columns if c.startswith('dis_') and c not in dis_cols and c != 'dis_imp2'] + ['SOH_target', 'RUL']
imp2_cols = ['dis_imp2', 'imp2_t_cycle', 'imp2_start', 'imp2_Re', 'imp2_Rct']
imp2_complex = [c for c in df_all.columns if c.startswith('imp2_') and c not in imp2_cols and c != 'imp2_cha']

final_col_order = base_cols + charge_cols + imp1_cols + imp1_complex + dis_cols + discharge_cols + imp2_cols + imp2_complex + ['imp2_cha', 'global_cycle']

df_all = df_all[final_col_order]

display(df_all)

,cha_t_cycle,cha_start,cha_time,cha_V_first,cha_V_last,cha_V_diff,cha_I_first,cha_I_last,cha_I_diff,cha_T_first,cha_T_mean,cha_T_last,cha_T_max,cha_T_diff,cha_V_charge_first,cha_V_charge_last,cha_V_charge_diff,cha_I_charge_first,cha_I_charge_last,cha_I_charge_diff,cha_imp,imp_t_cycle,imp_start,imp1_Re,imp1_Rct,imp_Sense_current_real,imp_Sense_current_imag,imp_Battery_current_real,imp_Battery_current_imag,imp_Current_ratio_real,imp_Current_ratio_imag,imp_Battery_impedance_real,imp_Battery_impedance_imag,imp_Rectified_Impedance_real,imp_Rectified_Impedance_imag,imp_dis,dis_t_cycle,dis_start,dis_time,dis_V_first,dis_V_last,dis_V_diff,dis_I_first,dis_I_last,dis_I_diff,dis_T_first,dis_T_mean,dis_T_last,dis_T_max,dis_T_diff,dis_I_load_mean,dis_V_load_mean,dis_Capacity,SOH_target,RUL,dis_imp2,imp2_t_cycle,imp2_start,imp2_Re,imp2_Rct,imp2_Sense_current_real,imp2_Sense_current_imag,imp2_Battery_current_real,imp2_Battery_current_imag,imp2_Current_ratio_real,imp2_Current_ratio_imag,imp2_Battery_impedance_real,imp2_Battery_impedance_imag,imp2_Rectified_Impedance_real,imp2_Rectified_Impedance_imag,imp2_cha,global_cycle
0,0,2008-04-02 13:08:17,7597.875,3.873017,4.191078,0.318060,-0.001201,-0.002892,-0.001692,24.655358,25.324079,24.507040,27.445134,-0.148317,0.003,0.003,0.000,0.000,0.000,0.000,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2008-04-02 15:25:41,3690.234,4.191492,3.277170,-0.914322,-0.004902,-0.006528,-0.001627,24.330034,32.572328,34.230853,38.982181,9.900819,-1.805570,2.404944,1.856487,100.000000,100,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2,2008-04-02 16:37:51,10516.000,3.325055,4.189062,0.864007,0.000302,-0.001833,-0.002135,29.341851,26.635623,24.949952,29.341949,-4.391898,0.003,0.003,0.000,-0.002,0.000,0.002,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,2008-04-02 19:43:48,3672.344,4.189773,3.300245,-0.889528,0.000021,-0.000448,-0.000469,24.697752,32.725235,34.392137,39.033398,9.694385,-1.804583,2.399260,1.846327,99.452721,99,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,4,2008-04-02 20:55:40,10484.547,3.352604,4.187398,0.834795,0.001990,0.002277,0.000287,29.553301,26.778176,24.977085,29.553301,-4.576215,0.003,0.003,0.000,0.000,0.000,0.000,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,2008-04-03 00:01:06,3651.641,4.188187,3.327451,-0.860736,-0.001754,0.001026,0.002780,24.734266,32.642862,34.232779,38.818797,9.498513,-1.803575,2.397969,1.835349,98.861386,98,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,6,2008-04-03 01:12:38,10397.890,3.378799,4.188055,0.809256,-0.002266,-0.000862,0.001404,29.456340,26.703204,24.895222,29.456340,-4.561118,0.003,0.003,0.000,0.000,0.000,0.000,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,2008-04-03 04:16:37,3631.563,4.188461,3.314182,-0.874279,-0.002775,-0.002234,0.000541,24.654236,32.514876,34.413450,38.762305,9.759213,-1.812863,2.408289,1.835263,98.856718,97,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,8,2008-04-03 05:27:49,10495.203,3.372871,4.188438,0.815568,-0.000503,0.000398,0.000901,29.481334,26.617004,24.676120,29.481334,-4.805213,0.003,0.003,0.000,0.000,0.000,0.000,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,2008-04-03 08:33:25,3629.172,4.188299,3.305497,-0.882802,-0.007981,0.000009,0.007990,24.524797,32.382349,34.345885,38.665393,9.821088,-1.812876,2.408505,1.834646,98.823482,96,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
5,10,2008-04-03 09:44:35,10792.672,3.366775,4.188695,0.821920,-0.000755,-0.004006,-0.003250,29.395820,26.518495,24.634213,29.395820,-4.761607,0.003,0.003,0.000,0.000,0.000,0.000,NaN,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,2008-04-03 12:55:10,3652.281,4.188816,3.302329,-0.886487,-0.005056,-0.003967,0.001088,24.522297,32.434182,34.200927,38.751695,9.678630,-1.803656,2.396251,1.835662,98.878217,95,NaN,<NA>,NaT,NaN,NaN,NaN,

In [18]:
df_all.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['cha_t_cycle', 'cha_start', 'cha_time', 'cha_V_first', 'cha_V_last',
       'cha_V_diff', 'cha_I_first', 'cha_I_last', 'cha_I_diff', 'cha_T_first',
       'cha_T_mean', 'cha_T_last', 'cha_T_max', 'cha_T_diff',
       'cha_V_charge_first', 'cha_V_charge_last', 'cha_V_charge_diff',
       'cha_I_charge_first', 'cha_I_charge_last', 'cha_I_charge_diff',
       'cha_imp', 'imp_t_cycle', 'imp_start', 'imp1_Re', 'imp1_Rct',
       'imp_Sense_current_real', 'imp_Sense_current_imag',
       'imp_Battery_current_real', 'imp_Battery_current_imag',
       'imp_Current_ratio_real', 'imp_Current_ratio_imag',
       'imp_Battery_impedance_real', 'imp_Battery_impedance_imag',
       'imp_Rectified_Impedance_real', 'imp_Rectified_Impedance_imag',
       'imp_dis', 'dis_t_cycle', 'dis_start', 'dis_time', 'dis_V_first',
       'dis_V_last', 'dis_V_diff', 'dis_I_first', 'dis_I_last', 'dis_I_diff',
       'dis_T_first', 'dis_T_mean', 'dis_T_last', 'dis_T_max', '

In [23]:
# [가변 필터링 설정 구역]
keep_charge = True     # Charge 섹션 포함 여부
keep_discharge = True  # Discharge 섹션 포함 여부
keep_impedance = False # Impedance 섹션 포함 여부

# 1. 루프 내부에서 컬럼 선별
filtered_rows = []
essential_keys = ['global_cycle', 'cha_t_cycle', 'imp_t_cycle', 'dis_t_cycle', 'imp2_t_cycle']

for row_dict in final_rows:
    new_row = {}
    
    # [A] 필수 키 무조건 보존
    for k in essential_keys: 
        if k in row_dict:
            new_row[k] = row_dict[k]

    # [B] 섹션별 선택적 추가
    if keep_charge:
        cha_cols = {k: v for k, v in row_dict.items() if k.startswith('cha_') and k not in essential_keys}
        new_row.update(cha_cols)
        
    if keep_discharge:
        dis_cols = {k: v for k, v in row_dict.items() if k.startswith('dis_') and k not in essential_keys}
        # SOH_target, RUL 추가
    for extra in ['SOH_target', 'RUL']:
        if extra in row_dict:
            dis_cols[extra] = row_dict[extra]
        new_row.update(dis_cols)
        
    if keep_impedance:
        # imp_ 혹은 imp2_ 혹은 imp1_ 형태만 수집 (dis_t_cycle 등 방전 지표 침범 방지)
        imp_cols = {k: v for k, v in row_dict.items() if (k.startswith('imp_') or k.startswith('imp2_') or 'imp1_' in k) and k not in essential_keys}
        new_row.update(imp_cols)
        
    # 루프의 마지막에 append
    filtered_rows.append(new_row)

# 2. DataFrame 생성
df_filter = pd.DataFrame(filtered_rows)

# 3. 컬럼 순서 강제 재정렬 로직
ordered_cols = []

# 1순위: 필수 키 (Cycle ID 및 시간)
ordered_cols += [c for c in essential_keys if c in df_filter.columns]

# 2순위: Charge 섹션 (알파벳순 정렬)
if keep_charge:
    cha_cols = sorted([c for c in df_filter.columns if c.startswith('cha_') and c not in essential_keys])
    ordered_cols += cha_cols

# 3순위: Discharge 섹션 (알파벳순 정렬)
if keep_discharge:
    dis_cols = sorted([c for c in df_filter.columns if c.startswith('dis_') and c not in essential_keys])
    # SOH_target, RUL 추가
    for extra in ['SOH_target', 'RUL']:
        if extra in df_filter.columns:
            dis_cols.append(extra)
    ordered_cols += dis_cols

# 4순위: Impedance 섹션
if keep_impedance:
    imp_cols = sorted([c for c in df_filter.columns if (c.startswith('imp') or 'imp' in c) and c not in essential_keys])
    ordered_cols += imp_cols

# 최종 정렬 적용
df_filter = df_filter[ordered_cols]

display(df_filter)

,global_cycle,cha_t_cycle,imp_t_cycle,dis_t_cycle,imp2_t_cycle,cha_I_charge_diff,cha_I_charge_first,cha_I_charge_last,cha_I_diff,cha_I_first,cha_I_last,cha_T_diff,cha_T_first,cha_T_last,cha_T_max,cha_T_mean,cha_V_charge_diff,cha_V_charge_first,cha_V_charge_last,cha_V_diff,cha_V_first,cha_V_last,cha_imp,cha_start,cha_time,dis_Capacity,dis_I_diff,dis_I_first,dis_I_last,dis_I_load_mean,dis_T_diff,dis_T_first,dis_T_last,dis_T_max,dis_T_mean,dis_V_diff,dis_V_first,dis_V_last,dis_V_load_mean,dis_imp2,dis_start,dis_time,SOH_target,RUL
0,1,0,<NA>,1,<NA>,0.000,0.000,0.000,-0.001692,-0.001201,-0.002892,-0.148317,24.655358,24.507040,27.445134,25.324079,0.000,0.003,0.003,0.318060,3.873017,4.191078,NaN,2008-04-02 13:08:17,7597.875,1.856487,-0.001627,-0.004902,-0.006528,-1.805570,9.900819,24.330034,34.230853,38.982181,32.572328,-0.914322,4.191492,3.277170,2.404944,NaN,2008-04-02 15:25:41,3690.234,100.000000,100
1,2,2,<NA>,3,<NA>,0.002,-0.002,0.000,-0.002135,0.000302,-0.001833,-4.391898,29.341851,24.949952,29.341949,26.635623,0.000,0.003,0.003,0.864007,3.325055,4.189062,NaN,2008-04-02 16:37:51,10516.000,1.846327,-0.000469,0.000021,-0.000448,-1.804583,9.694385,24.697752,34.392137,39.033398,32.725235,-0.889528,4.189773,3.300245,2.399260,NaN,2008-04-02 19:43:48,3672.344,99.452721,99
2,3,4,<NA>,5,<NA>,0.000,0.000,0.000,0.000287,0.001990,0.002277,-4.576215,29.553301,24.977085,29.553301,26.778176,0.000,0.003,0.003,0.834795,3.352604,4.187398,NaN,2008-04-02 20:55:40,10484.547,1.835349,0.002780,-0.001754,0.001026,-1.803575,9.498513,24.734266,34.232779,38.818797,32.642862,-0.860736,4.188187,3.327451,2.397969,NaN,2008-04-03 00:01:06,3651.641,98.861386,98
3,4,6,<NA>,7,<NA>,0.000,0.000,0.000,0.001404,-0.002266,-0.000862,-4.561118,29.456340,24.895222,29.456340,26.703204,0.000,0.003,0.003,0.809256,3.378799,4.188055,NaN,2008-04-03 01:12:38,10397.890,1.835263,0.000541,-0.002775,-0.002234,-1.812863,9.759213,24.654236,34.413450,38.762305,32.514876,-0.874279,4.188461,3.314182,2.408289,NaN,2008-04-03 04:16:37,3631.563,98.856718,97
4,5,8,<NA>,9,<NA>,0.000,0.000,0.000,0.000901,-0.000503,0.000398,-4.805213,29.481334,24.676120,29.481334,26.617004,0.000,0.003,0.003,0.815568,3.372871,4.188438,NaN,2008-04-03 05:27:49,10495.203,1.834646,0.007990,-0.007981,0.000009,-1.812876,9.821088,24.524797,34.345885,38.665393,32.382349,-0.882802,4.188299,3.305497,2.408505,NaN,2008-04-03 08:33:25,3629.172,98.823482,96
5,6,10,<NA>,11,<NA>,0.000,0.000,0.000,-0.003250,-0.000755,-0.004006,-4.761607,29.395820,24.634213,29.395820,26.518495,0.000,0.003,0.003,0.821920,3.366775,4.188695,NaN,2008-04-03 09:44:35,10792.672,1.835662,0.001088,-0.005056,-0.003967,-1.803656,9.678630,24.522297,34.200927,38.751695,32.434182,-0.886487,4.188816,3.302329,2.396251,NaN,2008-04-03 12:55:10,3652.281,98.878217,95
6,7,12,<NA>,13,<NA>,0.002,-0.002,0.000,-0.002386,0.002765,0.000380,-4.733952,29.438424,24.704472,29.438424,26.620706,0.000,0.003,0.003,0.827603,3.361036,4.188639,NaN,2008-04-03 14:06:43,10789.985,1.835146,-0.000791,-0.002633,-0.003425,-1.803621,9.713406,24.584878,34.298285,38.820701,32.480416,-0.894651,4.188392,3.293741,2.394646,NaN,2008-04-03 17:17:16,3650.828,98.850449,94
7,8,14,<NA>,15,<NA>,0.000,0.000,0.000,-0.002451,-0.000828,-0.003279,-4.484470,29.499916,25.015446,29.510811,26.737919,0.000,0.003,0.003,0.833766,3.353943,4.187709,NaN,2008-04-03 18:28:47,10127.562,1.825757,-0.000517,0.002228,0.001710,-1.830884,10.048613,24.713629,34.762242,38.517130,32.410462,-0.872697,4.188928,3.316231,2.434628,NaN,2008-04-03 21:28:14,3572.453,98.344690,93
8,9,16,<NA>,17,<NA>,0.000,0.000,0.000,-0.000338,0.000186,-0.000153,-4.807522,29.668814,24.861292,29.668814,26.741457,0.000,0.003,0.003,0.801985,3.386527,4.188513,NaN,2008-04-03 22:38:27,10147.953,1.824774,0.001192,-0.002196,-0.001004,-1.840505,10.416618,24.635212,35.051830,38.526268,32.346141,-0.891618,4.189029,3.297411,2.448511,NaN,2008-04-04 01:38:15,3550.594,98.291743,92
9,10,18,<NA>,19,<NA>,0.000,0.000,0.000,0.003267,-0.003345,-0.000077,-4.974435,29.779289,2

In [24]:
df_filter.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['global_cycle', 'cha_t_cycle', 'imp_t_cycle', 'dis_t_cycle',
       'imp2_t_cycle', 'cha_I_charge_diff', 'cha_I_charge_first',
       'cha_I_charge_last', 'cha_I_diff', 'cha_I_first', 'cha_I_last',
       'cha_T_diff', 'cha_T_first', 'cha_T_last', 'cha_T_max', 'cha_T_mean',
       'cha_V_charge_diff', 'cha_V_charge_first', 'cha_V_charge_last',
       'cha_V_diff', 'cha_V_first', 'cha_V_last', 'cha_imp', 'cha_start',
       'cha_time', 'dis_Capacity', 'dis_I_diff', 'dis_I_first', 'dis_I_last',
       'dis_I_load_mean', 'dis_T_diff', 'dis_T_first', 'dis_T_last',
       'dis_T_max', 'dis_T_mean', 'dis_V_diff', 'dis_V_first', 'dis_V_last',
       'dis_V_load_mean', 'dis_imp2', 'dis_start', 'dis_time', 'SOH_target',
       'RUL'],
      dtype='object')>